<a href="https://colab.research.google.com/github/Nithinkv/nifty50-predictions/blob/feature/paper-trading/train_models_colab_enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Enhanced NIFTY 50 LSTM Training (107 Months)

This notebook trains the **Enhanced LSTM Model** with **17 features** (including NSE data, Market Regime, and Options Proxies) for every month from 2016 to 2024.

**Features:**
1.  **Price/Volume**: Return, Vol_Change, Price_vs_MA20, Price_vs_MA50, BB_Position, Volatility, Momentum
2.  **Technical**: RSI_Scaled
3.  **NSE Data**: Deliverable_Scaled, Trade_Intensity_Scaled, Liquidity_Scaled, Volume_Surge_Scaled
4.  **Market Regime**: VIX_Scaled, Nifty_Trend, Nifty_Return_Scaled
5.  **Options Proxies**: GK_Vol_Scaled (Garman-Klass), MFI_Scaled (Smart Money)

**Training Strategy:**
-   **Walk-Forward**: Train on all history up to Month X, Predict Month X+1
-   **Custom Loss**: Directional Loss (penalizes wrong direction 2x)
-   **Sample Weighting**: Recency bias (recent data weighted 3x)
-   **Total Models**: 107 months × 50 stocks = 5350 training cycles (using global model per month)

In [ ]:
# Install dependencies
!pip install yfinance pandas numpy tensorflow scikit-learn nsepy

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from nsepy import get_history
from datetime import datetime, timedelta
import os
import glob

# Constants
LOOKBACK = 60
STOCKS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'KOTAKBANK.NS',
    'LT.NS', 'AXISBANK.NS', 'BAJFINANCE.NS', 'ASIANPAINT.NS', 'MARUTI.NS',
    'HCLTECH.NS', 'SUNPHARMA.NS', 'ULTRACEMCO.NS', 'TITAN.NS', 'NESTLEIND.NS',
    'WIPRO.NS', 'ONGC.NS', 'NTPC.NS', 'POWERGRID.NS', 'TATASTEEL.NS',
    'JSWSTEEL.NS', 'M&M.NS', 'TECHM.NS', 'INDUSINDBK.NS', 'ADANIENT.NS',
    'CIPLA.NS', 'DRREDDY.NS', 'TATAMOTORS.NS', 'BAJAJFINSV.NS', 'COALINDIA.NS',
    'EICHERMOT.NS', 'TATACONSUM.NS', 'GRASIM.NS', 'BRITANNIA.NS', 'HINDALCO.NS',
    'APOLLOHOSP.NS', 'BPCL.NS', 'DIVISLAB.NS', 'HEROMOTOCO.NS', 'ADANIPORTS.NS',
    'SBILIFE.NS', 'UPL.NS', 'BAJAJ-AUTO.NS', 'SHRIRAMFIN.NS', 'LTIM.NS'
]

# Ensure reproducible results
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# --- DATA FETCHING & FEATURE ENGINEERING ---

def fetch_market_regime_data(start_date, end_date):
    try:
        vix = yf.download("^INDIAVIX", start=start_date, end=end_date, progress=False)
        nifty = yf.download("^NSEI", start=start_date, end=end_date, progress=False)
        
        if isinstance(vix.columns, pd.MultiIndex): vix.columns = vix.columns.get_level_values(0)
        if isinstance(nifty.columns, pd.MultiIndex): nifty.columns = nifty.columns.get_level_values(0)
        
        regime_df = pd.DataFrame(index=nifty.index)
        regime_df['India_VIX'] = vix['Close'].reindex(nifty.index).fillna(method='ffill')
        regime_df['Nifty_Close'] = nifty['Close']
        regime_df['Nifty_MA50'] = nifty['Close'].rolling(window=50).mean()
        regime_df['Nifty_Trend'] = (regime_df['Nifty_Close'] > regime_df['Nifty_MA50']).astype(int)
        regime_df['Nifty_Return'] = nifty['Close'].pct_change()
        return regime_df
    except Exception as e:
        print(f"Regime fetch failed: {e}")
        return None

def fetch_enhanced_data(symbol, start_date, end_date):
    # 1. Fetch Basic Data
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        if df.empty: return None
    except: return None
    
    df = df.copy()
    
    # 2. Simulate NSE Features (Deliverable, Trade Intensity)
    daily_range = df['High'] - df['Low']
    close_pos = (df['Close'] - df['Low']) / daily_range.replace(0, np.nan)
    df['Deliverable_Pct'] = (close_pos * 50 + 25).fillna(50)
    
    vol_std = df['Volume'].rolling(20).std()
    vol_mean = df['Volume'].rolling(20).mean()
    df['Trade_Intensity'] = (vol_std / vol_mean.replace(0, np.nan) * 100).fillna(50)
    
    vol_rank = df['Volume'].rank(pct=True) * 100
    price_stab = 100 - (daily_range / df['Close'] * 100)
    df['Liquidity_Score'] = (vol_rank + price_stab) / 2
    
    df['Volume_MA20'] = df['Volume'].rolling(20).mean()
    df['Volume_Surge'] = (df['Volume'] / df['Volume_MA20'].replace(0, np.nan)).fillna(1)
    
    # 3. Advanced Volatility (Options Proxies)
    log_hl = np.log(df['High'] / df['Low'])**2
    log_co = np.log(df['Close'] / df['Open'])**2
    df['Garman_Klass_Vol'] = np.sqrt(0.5 * log_hl - (2 * np.log(2) - 1) * log_co)
    
    # 4. Money Flow Index (Smart Money)
    typical = (df['High'] + df['Low'] + df['Close']) / 3
    flow = typical * df['Volume']
    pos_flow = flow.where(typical > typical.shift(1), 0).rolling(14).mean()
    neg_flow = flow.where(typical < typical.shift(1), 0).rolling(14).mean()
    mfi_ratio = pos_flow / neg_flow
    df['MFI'] = 100 - (100 / (1 + mfi_ratio))
    
    # 5. Technicals
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    df['MA_20'] = df['Close'].rolling(20).mean()
    df['MA_50'] = df['Close'].rolling(50).mean()
    
    return df

In [ ]:
# --- PREPROCESSING ---

def prepare_data(df, regime_df, lookback=60):
    data = df.copy()
    
    # Merge Regime Data
    data = data.join(regime_df, how='left')
    data['India_VIX'].fillna(15.0, inplace=True)
    data['Nifty_Trend'].fillna(1, inplace=True)
    data['Nifty_Return'].fillna(0, inplace=True)
    
    # Calculate Target (Next Month Return)
    # We predict return 20 trading days ahead (approx 1 month)
    data['Target'] = data['Close'].shift(-20) / data['Close'] - 1
    
    # Feature Scaling
    data['Return'] = data['Close'].pct_change()
    data['Vol_Change'] = data['Volume'].pct_change().replace([np.inf, -np.inf], 0)
    
    data['RSI_Scaled'] = data['RSI'] / 100.0
    data['Deliverable_Scaled'] = data['Deliverable_Pct'] / 100.0
    data['Trade_Intensity_Scaled'] = np.clip(data['Trade_Intensity'] / 100.0, 0, 1)
    data['Liquidity_Scaled'] = data['Liquidity_Score'] / 100.0
    data['Volume_Surge_Scaled'] = np.clip(data['Volume_Surge'] / 3.0, 0, 1)
    
    data['Price_vs_MA20'] = (data['Close'] / data['MA_20'] - 1).clip(-0.2, 0.2)
    data['Price_vs_MA50'] = (data['Close'] / data['MA_50'] - 1).clip(-0.2, 0.2)
    
    bb_mid = data['Close'].rolling(20).mean()
    bb_std = data['Close'].rolling(20).std()
    data['BB_Position'] = ((data['Close'] - (bb_mid - 2*bb_std)) / (4*bb_std)).clip(0, 1)
    
    data['Volatility'] = data['Return'].rolling(20).std().clip(0, 0.1) * 10
    
    short_mom = data['Close'].pct_change(5)
    long_mom = data['Close'].pct_change(20)
    data['Momentum'] = (short_mom + long_mom).clip(-0.2, 0.2) / 0.4 + 0.5
    
    # Regime & Options Features
    data['VIX_Scaled'] = data['India_VIX'] / 30.0
    data['Nifty_Return_Scaled'] = data['Nifty_Return'] * 10
    data['GK_Vol_Scaled'] = data['Garman_Klass_Vol'] * 10
    data['MFI_Scaled'] = data['MFI'] / 100.0
    
    data.dropna(inplace=True)
    
    features = [
        'Return', 'Vol_Change', 'RSI_Scaled', 'Deliverable_Scaled', 
        'Trade_Intensity_Scaled', 'Liquidity_Scaled', 'Volume_Surge_Scaled',
        'Price_vs_MA20', 'Price_vs_MA50', 'BB_Position', 'Volatility', 'Momentum',
        'VIX_Scaled', 'Nifty_Trend', 'Nifty_Return_Scaled', 'GK_Vol_Scaled', 'MFI_Scaled'
    ]
    
    X, y = [], []
    dataset = data[features].values
    target = data['Target'].values
    dataset = np.clip(dataset, -1.0, 1.0)
    
    for i in range(lookback, len(dataset)):
        X.append(dataset[i-lookback:i])
        y.append(target[i])
        
    return np.array(X), np.array(y), data.index[lookback:]

In [ ]:
# --- MODEL DEFINITION ---

def directional_loss(y_true, y_pred):
    # Custom loss: Penalize wrong direction 2x
    mse = tf.square(y_true - y_pred)
    agreement = tf.sign(y_true) * tf.sign(y_pred)
    multiplier = tf.where(agreement >= 0, 1.0, 2.0)
    return tf.reduce_mean(mse * multiplier)

def create_model(input_shape):
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(50, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss=directional_loss, metrics=['mae'])
    return model

In [ ]:
# --- MAIN TRAINING LOOP ---

start_date = '2010-01-01'
end_date = '2024-12-31'

print("Fetching Market Regime Data...")
regime_df = fetch_market_regime_data(start_date, end_date)

print("Fetching Stock Data...")
stock_data = {}
for symbol in STOCKS:
    df = fetch_enhanced_data(symbol, start_date, end_date)
    if df is not None:
        stock_data[symbol] = df
        print(f"Loaded {symbol}: {len(df)} rows")

# Monthly Walk-Forward
months = pd.date_range(start='2016-01-01', end='2024-12-01', freq='MS')
results = []

for month in months:
    train_end = month - timedelta(days=1)
    test_end = month + timedelta(days=32)
    test_end = test_end.replace(day=1) - timedelta(days=1)
    
    print(f"\nProcessing Month: {month.strftime('%Y-%m')} (Train up to {train_end.date()})")
    
    # Prepare Global Training Data
    X_global, y_global = [], []
    weights_global = []
    
    for symbol, df in stock_data.items():
        # Filter data available up to training cutoff
        train_df = df[df.index <= train_end]
        if len(train_df) < LOOKBACK + 50: continue
        
        X, y, dates = prepare_data(train_df, regime_df)
        if len(X) == 0: continue
        
        X_global.append(X)
        y_global.append(y)
        
        # Sample Weights (Recency Bias)
        # Linear decay: Newest=1.5, Oldest=0.5
        w = np.linspace(0.5, 1.5, len(y))
        weights_global.append(w)
    
    if not X_global: continue
    
    X_train = np.concatenate(X_global)
    y_train = np.concatenate(y_global)
    w_train = np.concatenate(weights_global)
    
    # Train Model
    model = create_model((X_train.shape[1], X_train.shape[2]))
    early_stop = EarlyStopping(monitor='loss', patience=2)
    
    model.fit(
        X_train, y_train, 
        sample_weight=w_train,
        epochs=5, 
        batch_size=128, 
        verbose=0,
        callbacks=[early_stop]
    )
    
    # Save Model (or Predict immediately)
    # For this notebook, we'll save predictions to analyze
    
    # Predict for current month
    for symbol, df in stock_data.items():
        # Get data for prediction (need lookback window ending at train_end)
        pred_df = df[df.index <= train_end].tail(LOOKBACK)
        if len(pred_df) < LOOKBACK: continue
        
        # Prepare single sample
        # Note: We need to construct the features manually for the last window
        # Re-using prepare_data is inefficient but safe
        X_pred, _, _ = prepare_data(pred_df, regime_df, lookback=LOOKBACK-1)
        if len(X_pred) == 0: continue
        
        # Predict
        pred = model.predict(X_pred[-1:], verbose=0)[0][0]
        
        # Store result
        results.append({
            'Date': month,
            'Symbol': symbol,
            'Predicted_Return': pred
        })
        
    # Save intermediate results every year
    if month.month == 12:
        pd.DataFrame(results).to_csv('monthly_predictions_partial.csv')
        print("Saved partial results.")

# Final Save
res_df = pd.DataFrame(results)
res_df.to_csv('final_enhanced_predictions.csv', index=False)
print("Training Complete! Saved to final_enhanced_predictions.csv")